## 학습 데이터 출처
- 데이터: PhiUSIIL Phishing URL (Website), UCI Machine Learning Repository 번호 967
- 주소: https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset
- 만든 사람: Arvind Prasad, Shalini Chandra (2024)
- 논문: PhiUSIIL: A diverse security profile empowered phishing URL detection framework based on similarity index and incremental learning, Computers & Security, 2024
- DOI: https://doi.org/10.1016/j.cose.2023.103545
- 라이선스: CC BY 4.0 (출처를 밝히면 공유·변형 가능). 10/29 데이터 페이지에서 확인
- 주소 글자만 쓰고 그 주소에 접속하지 않는다. 데이터는 Colab 안에서만 쓰고 저장소에 올리지 않는다


In [ ]:
# 하는 일: UCI 저장소에서 이름에 'phish'가 들어간 데이터 목록을 찾아 보여 준다
# 이유: 데이터 이름을 정확히 몰라서 내려받기에 실패했으니, 저장소에 실제로 있는 이름과 번호(id)를 확인하려고
# 참고: 번호(id)로 내려받으면 이름 철자가 달라도 정확히 가져올 수 있다
from ucimlrepo import list_available_datasets
list_available_datasets(search="phish")

In [ ]:
# 하는 일: 학습에 쓸 공개 데이터(PhiUSIIL 피싱 URL 데이터셋)를 UCI 저장소에서 내려받는다
# 이유: AI 모델이 '피싱 주소'와 '정상 주소'의 차이를 배우려면 정답이 붙은 주소가 많이 필요하다
# 참고: 주소 '글자'만 쓰고 그 주소에 접속하지 않는다. 데이터는 Colab 안에서만 쓰고 저장소에는 올리지 않는다
!pip install -q ucimlrepo
from ucimlrepo import fetch_ucirepo

ds = fetch_ucirepo(id=967)   # PhiUSIIL Phishing URL (Website), UCI 저장소 번호 967
df = ds.data.original          # 데이터 전체를 표 모양으로 가져옴

print("크기 (줄, 칸):", df.shape)
print("칸 이름 앞부분:", df.columns.tolist()[:12])

In [ ]:
# 하는 일: 내려받은 데이터에 함께 들어 있는 설명 정보(출처, 만든 사람, 논문, 주소)를 보여 준다
# 이유: 작품설명서에 학습 데이터의 출처를 정확히 적어야 한다 (출처 미표기는 심사 제외 사유)
# 참고: 칸이 없으면 '없음'으로 나온다. 라이선스는 아래 주소의 데이터 페이지에서 한 번 더 눈으로 확인한다
m = ds.metadata
for k in ["name", "uci_id", "repository_url", "dataset_doi", "creators", "year_of_dataset_creation", "intro_paper"]:
    print(k, ":", m.get(k, "없음"))

In [ ]:
# 하는 일: 정답 칸(label)의 값별 개수와, 값마다 예시 주소 5개를 보여 준다
# 이유: 데이터마다 1이 피싱인지 정상인지 다르다. 사람이 예시를 보고 직접 판단해야 거꾸로 배우는 일을 막는다
# 참고: 피싱 주소가 섞여 있어서, 실수로 눌러도 열리지 않게 http는 hxxp로, 점(.)은 [.]으로 바꿔서 보여 준다
print("마지막 칸 이름:", df.columns[-1])
print(df["label"].value_counts())
for v in sorted(df["label"].unique()):
    print(f"\n[label = {v}] 예시 주소")
    for u in df[df["label"] == v]["URL"].sample(5, random_state=1):
        print("  ", u.replace("http", "hxxp").replace(".", "[.]"))

In [ ]:
# 하는 일: 피싱이면 1이 되는 정답 칸(is_phishing)을 만들고, 주소 글자에서 특징 10개를 뽑는다
# 이유: A가 확인한 대로 이 데이터는 label 0이 피싱이다. 우리 결과지는 '피싱 확률'이라 피싱=1로 뒤집어 둔다
# 참고: 특징은 서버에서도 똑같이 계산할 수 있게 주소 글자만 쓴다. 경로 길이처럼 데이터에 치우친 특징은 뺀다
import re
from urllib.parse import urlparse

df["is_phishing"] = 1 - df["label"]

FREE_HOSTS = ["ipfs", "firebaseapp", "web.app", "github.io", "000webhost", "weebly", "wix", "glitch.me", "pages.dev", "netlify"]
WORDS = ["login", "signin", "verify", "secure", "account", "update", "bank", "confirm", "password", "auth"]

def features(url):
    # 하는 일: 주소 하나를 숫자 10개로 바꾼다 (모델이 읽을 수 있는 모양)
    u = url if "://" in url else "http://" + url
    host = (urlparse(u).hostname or "").lower()
    low = url.lower()
    labels = [x for x in host.split(".") if x]
    return [
        len(host),                                              # 1 사이트 이름 길이
        host.count("."),                                        # 2 점 개수
        host.count("-"),                                        # 3 하이픈 개수
        sum(c.isdigit() for c in host),                         # 4 숫자 개수
        1 if re.fullmatch(r"[\d.]+", host) else 0,             # 5 숫자 주소(IP)인지
        max(len(labels) - 2, 0),                                # 6 하위 이름(서브도메인) 개수
        1 if "@" in urlparse(u).netloc else 0,                  # 7 @ 기호 (사이트 이름 앞부분에서만)
        sum(w in host for w in WORDS),                          # 8 유도 단어 (사이트 이름에서만, 경로 치우침 방지)
        1 if any(h in host for h in FREE_HOSTS) else 0,         # 9 무료 호스팅 주소인지
        max((len(x) for x in labels), default=0),               # 10 가장 긴 이름 조각의 길이
    ]

NAMES = ["사이트이름길이", "점", "하이픈", "숫자", "IP주소", "하위이름", "@기호", "유도단어", "무료호스팅", "긴조각"]
X = df["URL"].apply(features).tolist()

import pandas as pd
F = pd.DataFrame(X, columns=NAMES)
F["is_phishing"] = df["is_phishing"].values
print("특징 평균 (0=정상, 1=피싱)")
print(F.groupby("is_phishing").mean().round(2).T)

# 치우침 확인: 주소 뒤에 경로가 붙어 있는 비율
has_path = df["URL"].apply(lambda u: len(urlparse(u if "://" in u else "http://" + u).path.strip("/")) > 0)
print("\n경로가 붙은 주소 비율 (0=정상, 1=피싱)")
print(has_path.groupby(df["is_phishing"]).mean().round(3))

In [ ]:
# 하는 일: 특징을 '등록한 이름'(accounts.google.com이면 google) 중심으로 다시 만든다 (9개)
# 이유: 처음 특징은 정상 데이터가 www.이름.com 모양뿐이라 '이름이 길고 조각이 많으면 피싱'이라는 지름길을 배워,
#       구글 계정·사내 포털 같은 진짜 주소를 피싱으로 봤다 (셀 5에서 98%, 94% 확인)
# 참고: .example은 연습용이라 .com처럼 계산한다 (서버도 똑같이). 하위 이름은 숫자·하이픈만 본다
!pip install -q tldextract
import tldextract
EXT = tldextract.TLDExtract()

def features(url):
    u = url if "://" in url else "http://" + url
    host = (urlparse(u).hostname or "").lower()
    if host.endswith(".example"):
        host = host[: -len(".example")] + ".com"      # 연습 주소는 .com처럼
    e = EXT(host)
    name = e.domain                                    # 등록한 이름 (예: google)
    sub = ".".join(x for x in e.subdomain.split(".") if x and x != "www")   # 앞에 붙은 이름 (www 제외)
    return [
        len(name),                                     # 1 등록 이름 길이
        name.count("-"),                               # 2 등록 이름 하이픈
        sum(c.isdigit() for c in name),                # 3 등록 이름 숫자
        sub.count("-"),                                # 4 앞 이름 하이픈
        sum(c.isdigit() for c in sub),                 # 5 앞 이름 숫자
        1 if re.fullmatch(r"[\d.]+", host) else 0,    # 6 숫자 주소(IP)
        1 if "@" in urlparse(u).netloc else 0,         # 7 @ 기호
        sum(w in name for w in WORDS),                 # 8 유도 단어 (등록 이름에서만)
        1 if any(h in host for h in FREE_HOSTS) else 0,  # 9 무료 호스팅
    ]

NAMES = ["등록이름길이", "등록이름하이픈", "등록이름숫자", "앞이름하이픈", "앞이름숫자", "IP주소", "@기호", "유도단어", "무료호스팅"]
F = pd.DataFrame(df["URL"].apply(features).tolist(), columns=NAMES)
F["is_phishing"] = df["is_phishing"].values
print(F.groupby("is_phishing").mean().round(2).T)

In [ ]:
# 하는 일: 데이터를 학습용 80%와 시험용 20%로 나누고, 로지스틱 회귀 모델을 학습해 시험용으로 성능을 잰다
# 이유: 처음 보는 주소에서도 맞히는지 확인하려고. 로지스틱 회귀는 특징마다 '곱할 숫자'만 남아서 서버로 옮기기 쉽다
# 참고: 특징마다 크기가 달라서 평균·표준편차로 맞춘 뒤 학습한다 (서버도 같은 계산을 한다)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

Xtr, Xte, ytr, yte = train_test_split(F[NAMES].values, F["is_phishing"].values,
                                      test_size=0.2, random_state=42, stratify=F["is_phishing"].values)
sc = StandardScaler().fit(Xtr)
clf = LogisticRegression(max_iter=1000).fit(sc.transform(Xtr), ytr)
pred = clf.predict(sc.transform(Xte))

print("정확도:", round(accuracy_score(yte, pred), 3))
print("피싱이라고 한 것 중 진짜 피싱 (정밀도):", round(precision_score(yte, pred), 3))
print("진짜 피싱 중 잡아낸 비율 (재현율):", round(recall_score(yte, pred), 3))
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
print(f"정상을 피싱으로 잘못 봄: {fp}개 / 피싱을 놓침: {fn}개 (시험용 {len(yte)}개 중)")
print("\n특징별 무게 (+면 피싱 쪽, -면 정상 쪽)")
for n, w in sorted(zip(NAMES, clf.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"  {n}: {w:+.2f}")

In [ ]:
# 하는 일: 우리 게임 시나리오 링크와 진짜 사이트 주소를 모델에 넣어 피싱 확률을 본다
# 이유: 시험용 점수보다, 실제로 우리 사이트에 들어갈 주소에서 말이 되는 확률이 나오는지가 더 중요하다
# 참고: 진짜 로그인 페이지처럼 경로가 붙은 정상 주소를 일부러 넣어, 경로 치우침이 사라졌는지도 확인한다
tests = [
    ("S01 건강보험 (피싱)", "https://health-notice.example/check"),
    ("S04 텔레그램 (피싱)", "https://taiegram-login.example/verify"),
    ("S06 정부24 (피싱)", "https://gov24-support.example/app"),
    ("S07 사이버수사 (피싱)", "https://police-notice.example/case"),
    ("S09 코드저장소 (피싱)", "https://codehub-verify.example/login"),
    ("S05 사내 포털 (정상)", "https://portal.company.example/security"),
    ("S08 사내 우편실 (정상)", "https://portal.company.example/mailroom"),
    ("네이버 로그인 (정상)", "https://nid.naver.com/nidlogin.login"),
    ("정부24 (정상)", "https://www.gov.kr/portal/main"),
    ("구글 계정 (정상)", "https://accounts.google.com/signin"),
]
for name, url in tests:
    p = clf.predict_proba(sc.transform([features(url)]))[0][1]
    print(f"{name:22} 피싱 확률 {p:.0%}")

In [ ]:
# 하는 일: 모델이 전체적으로 피싱과 정상을 얼마나 잘 가르는지(AUC)와, 기준(%)마다 성능을 보여 준다
# 이유: 39%와 34%처럼 의미 없는 차이를 화면에 보여 주지 않으려고, '이 이상이면 말해도 된다'는 기준을 데이터로 정한다
# 참고: AUC는 0.5면 동전 던지기, 1이면 완벽. '정상 오의심'은 정상 주소 100개 중 피싱이라고 잘못 말한 개수
from sklearn.metrics import roc_auc_score
p_te = clf.predict_proba(sc.transform(Xte))[:, 1]
print("AUC:", round(roc_auc_score(yte, p_te), 3))
print("\n기준   정밀도   재현율   정상 오의심(100개 중)")
for t in [0.4, 0.5, 0.6, 0.7, 0.8]:
    hit = p_te >= t
    prec = (yte[hit] == 1).mean() if hit.any() else 0
    rec = hit[yte == 1].mean()
    fpr = hit[yte == 0].mean() * 100
    print(f"{t:.0%}    {prec:.2f}     {rec:.2f}     {fpr:.1f}")

In [ ]:
# 하는 일: 학습 결과(특징마다 곱할 숫자, 평균·표준편차)를 url_model.json 파일로 저장하고 내 컴퓨터로 내려받는다
# 이유: 서버에는 scikit-learn을 올릴 수 없어서(크기 제한), 이 숫자들만 가져가 같은 계산을 한다
# 참고: 저장한 숫자로 직접 계산한 확률이 모델과 똑같은지 마지막에 대조한다 (서버 계산식이 맞는지 미리 확인)
import json, math
model = {
    "name": "캐치피싱 URL 피싱 확률 모델 v1 (로지스틱 회귀, 등록 이름 중심 특징 9개)",
    "features": NAMES,
    "mean": [round(float(v), 6) for v in sc.mean_],
    "scale": [round(float(v), 6) for v in sc.scale_],
    "coef": [round(float(v), 6) for v in clf.coef_[0]],
    "intercept": round(float(clf.intercept_[0]), 6),
    "threshold": 0.6,   # 이 이상일 때만 화면에 확률을 보여 줌 (10/29 A 결정, 정상 오의심 100개 중 1.2개)
    "trained": "2026-10-29",
    "data": "PhiUSIIL Phishing URL (Website), UCI Machine Learning Repository 967, Prasad & Chandra (2024), CC BY 4.0",
    "test": {"count": int(len(yte)), "accuracy": round(accuracy_score(yte, pred), 3),
             "precision": round(precision_score(yte, pred), 3), "recall": round(recall_score(yte, pred), 3)},
    "note": "주소 글자만 사용(접속 안 함). .example은 .com처럼 계산. 판정이 아니라 보조 확률",
}
json.dump(model, open("url_model.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)

def prob(url):
    z = model["intercept"] + sum(c * (v - m) / s for v, m, s, c in zip(features(url), model["mean"], model["scale"], model["coef"]))
    return 1 / (1 + math.exp(-z))

print("저장한 숫자로 계산 vs 모델 (같아야 함)")
for name, url in tests:
    print(f"  {name:22} {prob(url):.1%}  /  {clf.predict_proba(sc.transform([features(url)]))[0][1]:.1%}")

from google.colab import files
files.download("url_model.json")